In [ ]:
from pathlib import Path

import polars as pl


fpath = Path.cwd().parent / "data" / "processed" / "dataset.parquet"

dset_raw = pl.read_parquet(fpath)

column_name = "date_payment"
dset = dset_raw.with_columns(
    pl.col(column_name).dt.year().alias("year")
)

min_year = 2010
max_year = 2025
dset_outliers = dset.filter(~pl.col("year").is_between(min_year, max_year))
print(f"Outside {min_year}-{max_year}: {dset_outliers.shape[0]} rows")
dset_stats = (
    dset_outliers.group_by("year")
        .agg(pl.len().alias("payments"))
        .sort("year")
)
print("Possible outliers")
display(dset_stats.to_pandas())

# distributions by data_source
dset_stats = (
    dset_outliers.group_by(["data_source"])
        .agg(pl.len().alias("payments"))
        .sort(["data_source"])
)
display(dset_stats.to_pandas())

dset = dset.filter(pl.col("year").is_between(min_year, max_year))
dset_stats = (
    dset.group_by("year")
        .agg(pl.len().alias("payments"))
        .sort("year")
)
display(dset_stats.to_pandas())